In [71]:
import pyodbc

print(pyodbc.drivers())

['SQL Server', 'SQL Server Native Client 11.0', 'B1CRHPROXY', 'ODBC Driver 13 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)', 'ODBC Driver 18 for SQL Server', 'SQL Server Native Client RDA 11.0', 'ODBC Driver 17 for SQL Server']


In [72]:
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from dotenv import load_dotenv
import os
import pyodbc

load_dotenv()

SERVER = os.getenv("DB_SERVER1")
DBNAME = os.getenv("DB_NAME1")
USER = os.getenv("DB_USER1")
PASS = os.getenv("DB_PASSWORD1")
ENCRYPT = os.getenv("DB_ENCRYPT", "yes")
TRUST = os.getenv("DB_TRUST_CERT", "yes")

SERVER = SERVER.replace("\\\\", "\\")  # normaliza


cnx = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={SERVER};"
    f"DATABASE={DBNAME};"
    f"UID={USER};"
    f"PWD={PASS};"
    f"Encrypt={ENCRYPT};TrustServerCertificate={TRUST};"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={quote_plus(cnx)}")

In [73]:
from sqlalchemy import text
try:
    with engine.connect() as conn:
        print("Ping:", conn.execute(text("SELECT 1")).scalar_one())
        print("Version:", conn.execute(text("SELECT @@VERSION")).scalar_one())
        print("✅ Conexión OK")
except Exception as e:
    print("❌ Error al conectar:", e)

Ping: 1
Version: Microsoft SQL Server 2017 (RTM) - 14.0.1000.169 (X64) 
	Aug 22 2017 17:04:49 
	Copyright (C) 2017 Microsoft Corporation
	Standard Edition (64-bit) on Windows Server 2019 Standard 10.0 <X64> (Build 17763: )

✅ Conexión OK


In [74]:
import pandas as pd
# 🔥 TEST 2: traer datos a pandas
query = "SELECT TOP 10 * FROM OITM"

df = pd.read_sql(query, engine)

df.head(5)

,ItemCode,ItemName,FrgnName,ItmsGrpCod,CstGrpCode,VatGourpSa,CodeBars,VATLiable,PrchseItem,SellItem,...,U_CI_15_Familia,U_CI_16_CaMeHP,U_CI_16_Ratio,U_CI_16_DiEjeSal,U_CI_14_DiExtRoMM,U_CI_14_NoCotCo,U_CI_14_AnchoMM,U_CI_14_SentR,U_CI_14_DiEjeIN,U_CI_14_TiRod
0,"# 111A-30"" X 1.5",NaN,NaN,100,-1,NaN,None,Y,Y,Y,...,None,None,None,None,None,None,None,None,None,None
1,"# 111A-30"" X 1.5""",BANDA TRANSPORTADORA CON ESPUMA Y GUIA,160,106,-1,,None,Y,Y,Y,...,None,None,None,None,None,None,None,None,None,None
2,"# 111A-54"" X 1.5",NaN,NaN,100,-1,NaN,None,Y,Y,Y,...,None,None,None,None,None,None,None,None,None,None
3,"# 111A-54"" X 1.5""",BANDA TRANSPORTADORA CON ESPUMA Y GUIA,160,106,-1,,None,Y,Y,Y,...,None,None,None,None,None,None,None,None,None,None
4,#108GL,BANDA TRANSPORTADORA 2PLY POLYCR57 WHITE PU GL...,160,106,-1,,None,Y,Y,Y,...,None,None,None,None,None,None,None,None,None,None


In [75]:
query = """
SELECT 
    T0.ItemCode,
    T0.ItemName,
    T0.OnHand,
    T0.AvgPrice,
    T0.LastPurDat,
    T0.ItmsGrpCod,
    T1.ItmsGrpNam
FROM OITM T0
INNER JOIN OITB T1
    ON T0.ItmsGrpCod = T1.ItmsGrpCod
WHERE 
    T1.ItmsGrpNam = 'EQUI.AUTO.Y CONTROL'
    AND T0.LastPurDat > '2021-01-01'
    AND T0.OnHand > 0
ORDER BY 
    T0.LastPurDat DESC
"""

df_equi_auto_control = pd.read_sql(query, engine)

df_equi_auto_control.head()

,ItemCode,ItemName,OnHand,AvgPrice,LastPurDat,ItmsGrpCod,ItmsGrpNam
0,3RT20361AL20,"contactor de potencia, AC-3e/AC-3, 51 A, 22 kW...",1.0,173.540000,2026-04-29,105,EQUI.AUTO.Y CONTROL
1,6GK19011BB102AA0,"""Industrial Ethernet FastConnect RJ45 Plug 180...",10.0,16.450000,2026-04-29,105,EQUI.AUTO.Y CONTROL
2,6SL32550AA004CA1,SINAMICS G120 Basic Operator Panel (BOP-2),8.0,42.620065,2026-04-29,105,EQUI.AUTO.Y CONTROL
3,6XV18402AH10,"Cable estándar Industrial Ethernet FC TP, GP 2...",70.0,1.860000,2026-04-29,105,EQUI.AUTO.Y CONTROL
4,US2:CQD370,Interruptor CQD 3P 70 A 480/277V 14 kA DIN,1.0,84.940000,2026-04-29,105,EQUI.AUTO.Y CONTROL


In [76]:

df_equi_auto_control.info()

<class 'pandas.DataFrame'>
RangeIndex: 412 entries, 0 to 411
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   ItemCode    412 non-null    str           
 1   ItemName    412 non-null    str           
 2   OnHand      412 non-null    float64       
 3   AvgPrice    412 non-null    float64       
 4   LastPurDat  412 non-null    datetime64[us]
 5   ItmsGrpCod  412 non-null    int64         
 6   ItmsGrpNam  412 non-null    str           
dtypes: datetime64[us](1), float64(2), int64(1), str(3)
memory usage: 22.7 KB


In [77]:
query = """
SELECT 
    ListNum,
    ListName
FROM OPLN
WHERE ListNum IN (14, 12, 13, 11)
ORDER BY ListNum
"""

df_listas_precios = pd.read_sql(query, engine)

df_listas_precios

,ListNum,ListName
0,11,Promociones Ecommerce - 11
1,12,B2B VIP - 12
2,13,B2C - 13
3,14,B2B Estandar - 14


In [78]:
query = """
WITH Precios AS (
    SELECT
        ItemCode,
        MAX(CASE WHEN PriceList = 14 THEN Price END) AS Precio_B2B_Estandar,
        MAX(CASE WHEN PriceList = 12 THEN Price END) AS Precio_B2B_VIP,
        MAX(CASE WHEN PriceList = 13 THEN Price END) AS Precio_B2C,
        MAX(CASE WHEN PriceList = 11 THEN Price END) AS Precio_Promociones_Ecommerce
    FROM ITM1
    WHERE PriceList IN (14, 12, 13, 11)
    GROUP BY ItemCode
),

Ventas AS (
    SELECT
        ItemCode,
        WhsCode,
        SUM(CASE WHEN YEAR(DocDate) = 2024 THEN Cantidad ELSE 0 END) AS Ventas_2024,
        SUM(CASE WHEN YEAR(DocDate) = 2025 THEN Cantidad ELSE 0 END) AS Ventas_2025,
        SUM(CASE WHEN YEAR(DocDate) = 2026 THEN Cantidad ELSE 0 END) AS Ventas_2026
    FROM (
        SELECT
            T1.ItemCode,
            T1.WhsCode,
            T0.DocDate,
            T1.Quantity AS Cantidad
        FROM OINV T0
        INNER JOIN INV1 T1
            ON T0.DocEntry = T1.DocEntry
        WHERE
            T0.CANCELED = 'N'
            AND YEAR(T0.DocDate) IN (2024, 2025, 2026)

        UNION ALL

        SELECT
            T1.ItemCode,
            T1.WhsCode,
            T0.DocDate,
            T1.Quantity * -1 AS Cantidad
        FROM ORIN T0
        INNER JOIN RIN1 T1
            ON T0.DocEntry = T1.DocEntry
        WHERE
            T0.CANCELED = 'N'
            AND YEAR(T0.DocDate) IN (2024, 2025, 2026)
    ) X
    GROUP BY
        ItemCode,
        WhsCode
)

SELECT 
    T0.ItemCode,
    T0.ItemName,
    T0.LastPurDat,
    T0.ItmsGrpCod,
    T1.ItmsGrpNam,

    T0.U_CI_CAT AS Categoria_CI,
    T0.U_CI_SCAT AS Subcategoria_CI,

    T3.WhsCode AS Bodega,
    T3.OnHand AS Stock_Bodega,

    P.Precio_B2B_Estandar,
    P.Precio_B2B_VIP,
    P.Precio_B2C,
    P.Precio_Promociones_Ecommerce,

    ISNULL(V.Ventas_2024, 0) AS Ventas_2024,
    ISNULL(V.Ventas_2025, 0) AS Ventas_2025,
    ISNULL(V.Ventas_2026, 0) AS Ventas_2026

FROM OITM T0

INNER JOIN OITB T1
    ON T0.ItmsGrpCod = T1.ItmsGrpCod

INNER JOIN OITW T3
    ON T0.ItemCode = T3.ItemCode

LEFT JOIN Precios P
    ON T0.ItemCode = P.ItemCode

LEFT JOIN Ventas V
    ON T0.ItemCode = V.ItemCode
    AND T3.WhsCode = V.WhsCode

WHERE 
    T1.ItmsGrpNam = 'EQUI.AUTO.Y CONTROL'
    AND T0.LastPurDat > '2021-01-01'
    AND T3.OnHand > 0

ORDER BY 
    T0.LastPurDat DESC,
    T0.ItemCode,
    T3.WhsCode
"""

df_equi_auto_control = pd.read_sql(query, engine)

df_equi_auto_control.head()

,ItemCode,ItemName,LastPurDat,ItmsGrpCod,ItmsGrpNam,Categoria_CI,Subcategoria_CI,Bodega,Stock_Bodega,Precio_B2B_Estandar,Precio_B2B_VIP,Precio_B2C,Precio_Promociones_Ecommerce,Ventas_2024,Ventas_2025,Ventas_2026
0,3RT20361AL20,"contactor de potencia, AC-3e/AC-3, 51 A, 22 kW...",2026-04-29,105,EQUI.AUTO.Y CONTROL,EQUIPO ELECTRICO & AUTOMATIZACION,CONTACTORES,11,1.0,168.7,159.40,187.50,0.0,27.0,43.0,13.0
1,6GK19011BB102AA0,"""Industrial Ethernet FastConnect RJ45 Plug 180...",2026-04-29,105,EQUI.AUTO.Y CONTROL,EQUIPO ELECTRICO & AUTOMATIZACION,OTROS,11,10.0,37.5,35.40,41.60,0.0,6.0,0.0,0.0
2,6SL32550AA004CA1,SINAMICS G120 Basic Operator Panel (BOP-2),2026-04-29,105,EQUI.AUTO.Y CONTROL,EQUIPO ELECTRICO & AUTOMATIZACION,VARIADORES,11,8.0,55.1,52.00,61.20,0.0,9.0,9.0,0.0
3,6XV18402AH10,"Cable estándar Industrial Ethernet FC TP, GP 2...",2026-04-29,105,EQUI.AUTO.Y CONTROL,EQUIPO ELECTRICO & AUTOMATIZACION,OTROS,11,70.0,4.0,3.75,4.45,0.0,0.0,0.0,0.0
4,US2:CQD370,Interruptor CQD 3P 70 A 480/277V 14 kA DIN,2026-04-29,105,EQUI.AUTO.Y CONTROL,EQUIPO ELECTRICO & AUTOMATIZACION,INTERRUPTORES INDUSTRIALES,11,1.0,119.8,113.20,133.10,0.0,4.0,3.0,4.0


In [79]:
df_equi_auto_control.info()

<class 'pandas.DataFrame'>
RangeIndex: 413 entries, 0 to 412
Data columns (total 16 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   ItemCode                      413 non-null    str           
 1   ItemName                      413 non-null    str           
 2   LastPurDat                    413 non-null    datetime64[us]
 3   ItmsGrpCod                    413 non-null    int64         
 4   ItmsGrpNam                    413 non-null    str           
 5   Categoria_CI                  359 non-null    str           
 6   Subcategoria_CI               359 non-null    str           
 7   Bodega                        413 non-null    str           
 8   Stock_Bodega                  413 non-null    float64       
 9   Precio_B2B_Estandar           413 non-null    float64       
 10  Precio_B2B_VIP                413 non-null    float64       
 11  Precio_B2C                    413 non-null 

## Análisis de alcance y riesgo de obsolescencia

A partir del resultado del query, se calcula la venta total acumulada del período 2024-2026, equivalente a 29.5 meses de análisis. Luego se calcula el alcance de inventario en meses por artículo y bodega.

El alcance se calcula como:

Stock actual de la bodega / Ventas acumuladas del período * 29.5

Cuando un artículo no tiene ventas en el período, el alcance se considera infinito, ya que existe inventario disponible pero no hay rotación histórica reciente.

In [80]:
# ============================================================
# CONFIGURACIÓN DE PARÁMETROS DEL ANÁLISIS
# ============================================================

MESES_ANALISIS = 29.5

# Umbrales de alcance de inventario
UMBRAL_ALCANCE_BAJO = 18
UMBRAL_ALCANCE_MEDIO = 36
UMBRAL_ALCANCE_ALTO = 36

# Umbrales de antigüedad de última compra
UMBRAL_ANIOS_COMPRA_ALTO = 2
UMBRAL_ANIOS_COMPRA_CRITICO = 3

In [81]:
# ============================================================
# PREPARACIÓN DEL DATAFRAME BASE
# ============================================================

df = df_equi_auto_control.copy()

# Convertir fecha de última compra
df["LastPurDat"] = pd.to_datetime(df["LastPurDat"], errors="coerce")

# Asegurar columnas numéricas
columnas_numericas = [
    "Stock_Bodega",
    "Ventas_2024",
    "Ventas_2025",
    "Ventas_2026",
    "Precio_B2B_Estandar",
    "Precio_B2B_VIP",
    "Precio_B2C",
    "Precio_Promociones_Ecommerce"
]

for col in columnas_numericas:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Calcular ventas totales del período
df["Ventas_Total_29_5_Meses"] = (
    df["Ventas_2024"] +
    df["Ventas_2025"] +
    df["Ventas_2026"]
)

# Calcular alcance de inventario en meses
df["Alcance_Inventario_Meses"] = np.where(
    df["Ventas_Total_29_5_Meses"] > 0,
    (df["Stock_Bodega"] / df["Ventas_Total_29_5_Meses"]) * MESES_ANALISIS,
    np.inf
)

# Calcular años desde última compra
hoy = pd.Timestamp.today()

df["Anios_Desde_Ultima_Compra"] = (
    (hoy - df["LastPurDat"]).dt.days / 365
).round(1)

In [82]:
# ============================================================
# ALCANCE REDONDEADO PARA REPORTE
# ============================================================

df["Alcance_Inventario_Meses_Redondeado"] = "Infinito"

mask_con_ventas = df["Ventas_Total_29_5_Meses"] > 0

df.loc[
    mask_con_ventas,
    "Alcance_Inventario_Meses_Redondeado"
] = (
    df.loc[mask_con_ventas, "Alcance_Inventario_Meses"]
    .round(0)
    .astype(int)
    .astype(str)
)

In [83]:
df.head()

,ItemCode,ItemName,LastPurDat,ItmsGrpCod,ItmsGrpNam,Categoria_CI,Subcategoria_CI,Bodega,Stock_Bodega,Precio_B2B_Estandar,Precio_B2B_VIP,Precio_B2C,Precio_Promociones_Ecommerce,Ventas_2024,Ventas_2025,Ventas_2026,Ventas_Total_29_5_Meses,Alcance_Inventario_Meses,Anios_Desde_Ultima_Compra,Alcance_Inventario_Meses_Redondeado
0,3RT20361AL20,"contactor de potencia, AC-3e/AC-3, 51 A, 22 kW...",2026-04-29,105,EQUI.AUTO.Y CONTROL,EQUIPO ELECTRICO & AUTOMATIZACION,CONTACTORES,11,1.0,168.7,159.40,187.50,0.0,27.0,43.0,13.0,83.0,0.355422,0.0,0
1,6GK19011BB102AA0,"""Industrial Ethernet FastConnect RJ45 Plug 180...",2026-04-29,105,EQUI.AUTO.Y CONTROL,EQUIPO ELECTRICO & AUTOMATIZACION,OTROS,11,10.0,37.5,35.40,41.60,0.0,6.0,0.0,0.0,6.0,49.166667,0.0,49
2,6SL32550AA004CA1,SINAMICS G120 Basic Operator Panel (BOP-2),2026-04-29,105,EQUI.AUTO.Y CONTROL,EQUIPO ELECTRICO & AUTOMATIZACION,VARIADORES,11,8.0,55.1,52.00,61.20,0.0,9.0,9.0,0.0,18.0,13.111111,0.0,13
3,6XV18402AH10,"Cable estándar Industrial Ethernet FC TP, GP 2...",2026-04-29,105,EQUI.AUTO.Y CONTROL,EQUIPO ELECTRICO & AUTOMATIZACION,OTROS,11,70.0,4.0,3.75,4.45,0.0,0.0,0.0,0.0,0.0,inf,0.0,Infinito
4,US2:CQD370,Interruptor CQD 3P 70 A 480/277V 14 kA DIN,2026-04-29,105,EQUI.AUTO.Y CONTROL,EQUIPO ELECTRICO & AUTOMATIZACION,INTERRUPTORES INDUSTRIALES,11,1.0,119.8,113.20,133.10,0.0,4.0,3.0,4.0,11.0,2.681818,0.0,3


In [84]:
# ============================================================
# DEFINICIÓN DE CRITERIOS DE RIESGO DE OBSOLESCENCIA
# ============================================================

# CRÍTICO:
# Artículos con stock, sin ventas en los últimos 29.5 meses
# y con última compra mayor o igual a 3 años.
# Interpretación:
# Alta probabilidad de material sin rotación reciente.
criterio_critico = (
    (df["Stock_Bodega"] > 0) &
    (df["Ventas_Total_29_5_Meses"] == 0) &
    (df["Anios_Desde_Ultima_Compra"] >= UMBRAL_ANIOS_COMPRA_CRITICO)
)

# ALTO:
# Artículos con stock, con ventas históricas, pero con alcance mayor a 36 meses
# y última compra mayor o igual a 2 años.
# Interpretación:
# Tiene algo de movimiento, pero el stock actual es demasiado alto contra la demanda.
criterio_alto = (
    (df["Stock_Bodega"] > 0) &
    (df["Ventas_Total_29_5_Meses"] > 0) &
    (df["Alcance_Inventario_Meses"] > UMBRAL_ALCANCE_ALTO) &
    (df["Anios_Desde_Ultima_Compra"] >= UMBRAL_ANIOS_COMPRA_ALTO)
)

# MEDIO:
# Artículos con ventas y alcance entre 18 y 36 meses.
# Interpretación:
# No necesariamente obsoleto, pero debe monitorearse.
criterio_medio = (
    (df["Stock_Bodega"] > 0) &
    (df["Ventas_Total_29_5_Meses"] > 0) &
    (df["Alcance_Inventario_Meses"] > UMBRAL_ALCANCE_BAJO) &
    (df["Alcance_Inventario_Meses"] <= UMBRAL_ALCANCE_MEDIO)
)

# BAJO:
# Artículos con ventas y alcance menor o igual a 18 meses.
# Interpretación:
# Inventario con rotación razonable.
criterio_bajo = (
    (df["Stock_Bodega"] > 0) &
    (df["Ventas_Total_29_5_Meses"] > 0) &
    (df["Alcance_Inventario_Meses"] <= UMBRAL_ALCANCE_BAJO)
)

In [85]:
# ============================================================
# CLASIFICACIÓN FINAL DE RIESGO DE OBSOLESCENCIA
# ============================================================

condiciones = [
    criterio_critico,
    criterio_alto,
    criterio_medio,
    criterio_bajo
]

clasificaciones = [
    "Crítico - Candidato a donación/liquidación",
    "Alto - Revisar para liquidación",
    "Medio - Monitorear / promover venta",
    "Bajo - Inventario saludable"
]

df["Riesgo_Obsolescencia"] = np.select(
    condiciones,
    clasificaciones,
    default="Revisar manualmente"
)

In [86]:
# ============================================================
# ACCIÓN RECOMENDADA
# ============================================================

df["Accion_Recomendada"] = np.select(
    [
        df["Riesgo_Obsolescencia"] == "Crítico - Candidato a donación/liquidación",
        df["Riesgo_Obsolescencia"] == "Alto - Revisar para liquidación",
        df["Riesgo_Obsolescencia"] == "Medio - Monitorear / promover venta",
        df["Riesgo_Obsolescencia"] == "Bajo - Inventario saludable"
    ],
    [
        "Validar técnicamente y considerar donación, liquidación o depuración",
        "Revisar precio, compatibilidad y posibilidad de promoción agresiva",
        "Mantener en observación e impulsar venta comercialmente",
        "Mantener en inventario"
    ],
    default="Revisión manual"
)

In [87]:
df.to_csv(
    "Clasificacion_automatizada_riesgo_obsolescencia.csv",
    index=False,
    encoding="utf-8-sig"
)